# **1. Perkenalan Dataset**

Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.


**Dataset yang digunakan: UCI Wine Quality (White Wine).**

Dataset ini berasal dari [UCI Machine Learning Repository - Wine Quality](https://archive.ics.uci.edu/ml/datasets/wine+quality). Dataset memuat **4,898 baris** dan **12 kolom** (11 fitur + 1 target) dengan header, dengan pemisah semicolon (;).

Fitur-fiturnya meliputi: `fixed acidity`, `volatile acidity`, `citric acid`, `residual sugar`, `chlorides`, `free sulfur dioxide`, `total sulfur dioxide`, `density`, `pH`, `sulphates`, `alcohol`, dan `quality`.

Kolom `quality` aslinya bernilai 0-10 (skor kualitas wine).

Untuk pengerjaan submission ini, target dibinarisasi menjadi **klasifikasi biner**: `quality >= 6` → 1 (good), dan `< 6` → 0 (bad).

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import importlib.util
from pathlib import Path

%matplotlib inline

RAW_FILENAME = "wine-quality-white_raw.csv"
PROCESSED_FILENAME = "wine-quality-white_preprocessing.csv"


def locate_preprocessing_dir() -> Path:
    """Cari folder yang memuat berkas dataset mentah Wine Quality."""
    search_spots = [Path.cwd(), Path.cwd() / "preprocessing"]
    for spot in search_spots:
        if (spot / RAW_FILENAME).exists():
            return spot.resolve()
    # Cadangan: berkas mentah berada satu level di atas folder kerja.
    if (Path.cwd().parent / RAW_FILENAME).exists():
        return Path.cwd().resolve()
    raise FileNotFoundError(
        f"folder preprocessing berisi '{RAW_FILENAME}' tidak ditemukan "
        "(jalankan notebook ini dari dalam folder preprocessing/)"
    )


PREPROCESSING_DIR = locate_preprocessing_dir()
RAW_PATH = PREPROCESSING_DIR.parent / RAW_FILENAME
OUTPUT_PATH = PREPROCESSING_DIR / PROCESSED_FILENAME

_automate_path = PREPROCESSING_DIR / "automate_Devani.py"
_spec = importlib.util.spec_from_file_location("automate_preprocessing", _automate_path)
automate = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(automate)

DATASET_NAME = automate.DATASET_NAME
TARGET_COLUMN = automate.TARGET_COLUMN
preprocess_dataset = automate.preprocess_dataset

print("Versi pandas        :", pd.__version__)
print("Versi numpy         :", np.__version__)
print("Folder preprocessing:", PREPROCESSING_DIR)
print("Nama dataset        :", DATASET_NAME)
print("Kolom target        :", TARGET_COLUMN)

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
try:
    df_raw = pd.read_csv(RAW_PATH, sep=";")
except Exception as exc:
    raise RuntimeError(
        f"gagal membaca dataset dari {RAW_PATH!r}: {exc}"
    ) from exc

jumlah_baris, jumlah_kolom = df_raw.shape
print(f"Dataset termuat -> {jumlah_baris} baris, {jumlah_kolom} kolom")

df_raw.head()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
# dimensi dataset serta tipe data setiap kolom.
baris, kolom = df_raw.shape
print(f"Ukuran dataset : {baris} baris x {kolom} kolom")
print("\nTipe data tiap kolom:")
for nama_kolom, tipe in df_raw.dtypes.items():
    print(f"  - {nama_kolom:<22}: {tipe}")

print("\nRingkasan struktur (df.info):")
df_raw.info()

In [ ]:
# Statistik deskriptif tiap fitur numerik (dibulatkan agar mudah dibaca).
df_raw.describe().T.round(3)

In [ ]:
# Hitung nilai yang hilang pada setiap kolom.
nilai_hilang = df_raw.isna().sum()
total_hilang = int(nilai_hilang.sum())

print("Nilai hilang per kolom:")
kolom_bermasalah = nilai_hilang[nilai_hilang > 0]
if kolom_bermasalah.empty:
    print("  (tidak ada kolom dengan nilai hilang)")
else:
    print(kolom_bermasalah.to_string())
print(f"\nTotal nilai hilang di seluruh dataset: {total_hilang}")

In [ ]:
# Periksa keberadaan baris yang terduplikasi.
jumlah_duplikat = int(df_raw.duplicated().sum())
persentase_duplikat = jumlah_duplikat / len(df_raw) * 100

print(f"Baris duplikat : {jumlah_duplikat} dari {len(df_raw)} baris")
print(f"Proporsi       : {persentase_duplikat:.2f}%")

In [ ]:
# Bandingkan sebaran skor quality asli dengan versi binernya.
skor_asli = df_raw[TARGET_COLUMN].value_counts().sort_index()
label_biner = (df_raw[TARGET_COLUMN] >= 6).astype(int)
hitung_biner = label_biner.value_counts().sort_index()

fig, (ax_kiri, ax_kanan) = plt.subplots(1, 2, figsize=(11, 4.5))

ax_kiri.bar(skor_asli.index.astype(str), skor_asli.values, color="steelblue", edgecolor="black")
ax_kiri.set_title("Sebaran Skor Quality Asli (0-10)")
ax_kiri.set_xlabel("Skor Quality")
ax_kiri.set_ylabel("Jumlah Sampel")

ax_kanan.bar(["Bad (< 6)", "Good (>= 6)"], [hitung_biner[0], hitung_biner[1]],
             color=["indianred", "mediumseagreen"], edgecolor="black")
ax_kanan.set_title("Sebaran Target Setelah Binarisasi")
ax_kanan.set_ylabel("Jumlah Sampel")

fig.tight_layout()
plt.show()

total = len(df_raw)
print("Komposisi target biner:")
print(f"  Bad  (< 6) : {hitung_biner[0]:>4} sampel ({hitung_biner[0] / total * 100:.1f}%)")
print(f"  Good (>= 6): {hitung_biner[1]:>4} sampel ({hitung_biner[1] / total * 100:.1f}%)")

In [ ]:
# heatmap korelasi antar fitur menggunakan heatmap segitiga bawah.
matriks_korelasi = df_raw.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    matriks_korelasi,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    square=True,
)
plt.title("Heatmap Korelasi Antar Fitur Wine Quality")
plt.tight_layout()
plt.show()

# Urutkan korelasi tiap fitur terhadap target quality.
korelasi_target = matriks_korelasi[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(ascending=False)
print("Korelasi fitur terhadap 'quality' (tertinggi ke terendah):")
print(korelasi_target.round(3).to_string())

In [ ]:
# Tampilkan distribusi beberapa fitur yang paling berpengaruh.
fitur_penting = ["alcohol", "density", "volatile acidity", "chlorides"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, fitur in zip(axes.ravel(), fitur_penting):
    ax.hist(df_raw[fitur], bins=30, color="slateblue", edgecolor="black", alpha=0.75)
    ax.set_title(f"Distribusi '{fitur}'")
    ax.set_xlabel(fitur)
    ax.set_ylabel("Frekuensi")

fig.suptitle("Sebaran Fitur Penting pada Dataset Wine Quality", y=1.02)
fig.tight_layout()
plt.show()

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
# Buat salinan dataframe untuk preprocessing
df_processed = df_raw.copy()

print("=== PREPROCESSING STEPS ===")
print(f"\n1. Data awal: {df_processed.shape[0]} baris x {df_processed.shape[1]} kolom")

# Step 1: Pastikan semua kolom numerik
for col in df_processed.columns:
    df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
print(f"\n2. Setelah konversi numerik: {df_processed.shape[0]} baris")

# Step 2: Binarisasi target (quality >= 6 -> 1, < 6 -> 0)
original_quality = df_processed[TARGET_COLUMN].copy()
df_processed[TARGET_COLUMN] = (df_processed[TARGET_COLUMN].fillna(0) >= 6).astype(int)
print(f"\n3. Target dibinarisasi:")
print(f"   - Good (>= 6): {(df_processed[TARGET_COLUMN] == 1).sum()}")
print(f"   - Bad (< 6): {(df_processed[TARGET_COLUMN] == 0).sum()}")

# Step 3: Imputasi median untuk missing values (jika ada)
feature_cols = [c for c in df_processed.columns if c != TARGET_COLUMN]
missing_before = df_processed[feature_cols].isna().sum().sum()
for col in feature_cols:
    median = df_processed[col].median()
    if pd.isna(median):
        median = 0.0
    df_processed[col] = df_processed[col].fillna(median)
missing_after = df_processed.isna().sum().sum()
print(f"\n4. Missing values: {missing_before} -> {missing_after}")

# Step 4: Drop duplikat
n_before_drop = len(df_processed)
df_processed = df_processed.drop_duplicates().reset_index(drop=True)
n_after_drop = len(df_processed)
print(f"\n5. Drop duplikat: {n_before_drop} -> {n_after_drop} baris (removed {n_before_drop - n_after_drop})")

# Step 5: Pastikan semua kolom numerik
df_processed = df_processed.apply(pd.to_numeric)

print(f"\n=== HASIL PREPROCESSING ===")
print(f"Shape final: {df_processed.shape[0]} baris x {df_processed.shape[1]} kolom")
print(f"Missing values: {df_processed.isna().sum().sum()}")
print(f"Duplikat: {df_processed.duplicated().sum()}")
print(f"\nTarget balance:")
print(df_processed[TARGET_COLUMN].value_counts())

In [ ]:
# Verifikasi bahwa hasil manual sama dengan hasil dari fungsi automation
df_automated = preprocess_dataset(df_raw, output_path=None)

print("=== VERIFIKASI ===")
print(f"Shape manual   : {df_processed.shape}")
print(f"Shape automated: {df_automated.shape}")
print(f"\nApakah hasil identik? {df_processed.equals(df_automated)}")

if not df_processed.equals(df_automated):
    print("\nPerbedaan ditemukan:")
    for col in df_processed.columns:
        if not df_processed[col].equals(df_automated[col]):
            print(f"  - Kolom '{col}' berbeda")

In [ ]:
# Simpan dataset hasil preprocessing
df_processed.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset preprocessing berhasil disimpan ke: {OUTPUT_PATH}")
print(f"\nRingkasan:")
print(f"- Baris: {len(df_processed)}")
print(f"- Kolom: {len(df_processed.columns)}")
print(f"- Missing: {df_processed.isna().sum().sum()}")
print(f"- Target balance: {df_processed[TARGET_COLUMN].value_counts().to_dict()}")

In [ ]:
# Perbandingan distribusi target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sebelum (quality score asli)
axes[0].hist(original_quality, bins=range(0, 11), edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_title('Target Sebelum Preprocessing\n(Quality Score 0-10)')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].set_xticks(range(0, 11))

# Sesudah (binary: 0=bad, 1=good)
counts = df_processed[TARGET_COLUMN].value_counts().sort_index()
axes[1].bar(['Bad (< 6)', 'Good (>= 6)'], [counts[0], counts[1]], color=['coral', 'lightgreen'], edgecolor='black')
axes[1].set_title('Target Sesudah Preprocessing\n(Binary Classification)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("\n=== DATASET SIAP UNTUK MODELLING ===")